<a href="https://colab.research.google.com/github/genaiconference/Agentic_KAG_Workshop_DHS_2026/blob/main/05_personalised_movie_recommendation_using_living_graph_memory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎬 Graphiti-Native Living Graph — Personalized Movie Recommendation Agent
## powered by **Graphiti's own** memory engine + Neo4j GraphRAG Hybrid Retrieval

---

## 1. Capstone Overview

This notebook builds a **personalized movie recommendation agent** whose living memory is powered
entirely by [`graphiti-core`](https://github.com/getzep/graphiti) — a temporally-aware
knowledge-graph memory engine.

Graphiti handles the entire living-memory layer natively:

| Capability | How it works |
|-----------|--------------|
| Preference extraction | `graphiti.add_episode()` LLM extraction |
| Entity typing | Custom **Pydantic entity types** |
| Relationship typing | Custom **Pydantic edge types** + `edge_type_map` |
| Bitemporal facts | Graphiti fact edges (`valid_at`, `invalid_at`) |
| Invalidation on contradiction | Graphiti automatic edge invalidation |
| Episodic provenance | Graphiti episodes + `MENTIONS` |
| Memory search | `graphiti.search()` / `graphiti._search()` recipes |
| Per-user isolation | Graphiti `group_id` namespacing |
| Communities | `graphiti.build_communities()` |

### The other moving parts
- The **Neo4j Movie Intelligence Graph** and the **GraphRAG `HybridRetriever`** provide movie recall.
- A **LangChain ReAct agent** orchestrates five tools: `search_memory`, `update_memory`, `taste_profile`,
  `movie_hybrid_retriever`, `web_search`.

### Why this matters
Because Graphiti owns the living graph, the user's memory is **LLM-powered, schema-typed, auditable, and
bitemporal by construction** — every belief is typed, timestamped, and preserved across contradictions.


## 2. Architecture

The agent sits between the **user** and two graphs: **Graphiti's living user-memory graph** and the
**Neo4j Movie Intelligence Graph**. It orchestrates four tools to produce a personalized recommendation.

```mermaid
flowchart TD
    U[User Message] --> A[🤖 ReAct Recommendation Agent]

    A -->|1. always first| SM[search_memory tool]
    A -->|2. if new signal| UM[update_memory tool]
    A -->|3. on request| MR[movie_hybrid_retriever tool]
    A -->|4. if metadata missing| WS[web_search tool]

    SM -->|graphiti.search / _search| G[(🧠 Graphiti Living Memory<br/>bitemporal, typed, group_id)]
    UM -->|graphiti.add_episode| G
    MR <-->|GraphRAG HybridRetriever| MIG[(🎞️ Neo4j Movie<br/>Intelligence Graph)]
    WS <--> NET[(🌐 Tavily Web Search)]

    A --> REC[✅ Personalized<br/>Recommendation]
```

### The tools and what they call

| Tool | Backing call |
|------|-------------|
| **`search_memory`** | `graphiti.search()` + `_search()` recipes |
| **`update_memory`** | `graphiti.add_episode()` (LLM extraction + auto-invalidation) |
| **`taste_profile`** | `graphiti.build_communities()` clusters + summaries |
| **`movie_hybrid_retriever`** | Neo4j GraphRAG `HybridRetriever` |
| **`web_search`** | Tavily |

### Agent flow

```text
User Message
   ↓
search_memory                       ← graphiti.search(), always first
   ↓
if new signal → update_memory       ← graphiti.add_episode() extracts + invalidates
   ↓
if user asks for a recommendation:
    enough context? → No → ask ONE short clarifying question
                    → Yes → movie_hybrid_retriever
                            → web_search (only if metadata missing)
                            → recommend with reasons
else:
    acknowledge naturally
```


## 3. Environment Setup — Clone & Install

> 🛠️ **Run once (Colab-friendly).** Clone the workshop repo and install its dependencies
> (Graphiti, Neo4j GraphRAG, LangChain, Tavily). On a local machine skip the clone and install from
> your local `requirements.txt`.


In [1]:
!git clone https://github.com/genaiconference/Agentic_KAG_Workshop_DHS_2026.git

Cloning into 'Agentic_KAG_Workshop_DHS_2026'...
remote: Enumerating objects: 102, done.
remote: Counting objects: 100% (102/102), done.
remote: Compressing objects: 100% (99/99), done.
remote: Total 102 (delta 51), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (102/102), 2.90 MiB | 5.32 MiB/s, done.
Resolving deltas: 100% (51/51), done.


In [1]:
!pip install -r /content/Agentic_KAG_Workshop_DHS_2026/requirements.txt --quiet

## 4. Configure Credentials & Import Dependencies

This notebook requires **Neo4j** (hosting both the Movie Intelligence Graph and Graphiti's living-memory
graph), an **OpenAI** API key (Graphiti extraction/embeddings + the ReAct agent + GraphRAG), and a
**Tavily** web-search key. Replace all placeholder credentials with your own before running.

> 🔑 **Tip:** store secrets in a `.env` file (loaded via `python-dotenv`) rather than hard-coding them.


In [2]:
import os

# On Colab, switch into the cloned repo (harmless if it doesn't exist locally).
if os.path.isdir('/content/Agentic_KAG_Workshop_DHS_2026/'):
    os.chdir('/content/Agentic_KAG_Workshop_DHS_2026/')

try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    print("error reading env details")

# --- Neo4j ---
NEO4J_URI      = os.getenv('NEO4J_URI')
NEO4J_USERNAME = os.getenv('NEO4J_USERNAME')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD')
NEO4J_DATABASE = os.getenv('NEO4J_DATABASE', 'neo4j')

# --- OpenAI ---
os.environ.setdefault('OPENAI_API_KEY', os.getenv('OPENAI_API_KEY') or '')
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

# --- Tavily web search ---
WEB_SEARCH_API_KEY = os.getenv('WEB_SEARCH_API_KEY')

print('NEO4J_URI      :', NEO4J_URI)
print('OPENAI key set :', bool(os.environ.get('OPENAI_API_KEY')))
print('Tavily key set :', bool(WEB_SEARCH_API_KEY))

NEO4J_URI      : bolt://3.82.224.106
OPENAI key set : True
Tavily key set : True


In [3]:
# ---------------------------------------------------------------------------
# Core imports
# ---------------------------------------------------------------------------
import json
import asyncio
import logging
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional

import nest_asyncio
import pandas as pd
from IPython.display import display, Markdown

# Silence harmless Neo4j cold-start notifications (e.g. "property 'fact_embedding'
# does not exist") that appear only while the memory graph is still empty. Real
# errors are raised as exceptions and are NOT affected by this.
logging.getLogger("neo4j.notifications").setLevel(logging.ERROR)

# Allow asyncio.run-style calls inside the already-running notebook event loop.
nest_asyncio.apply()

def run_async(coro):
    """Run an async Graphiti coroutine from sync notebook code."""
    return asyncio.get_event_loop().run_until_complete(coro)

def utc_now() -> datetime:
    return datetime.now(timezone.utc)

# LLM model used across Graphiti extraction and the ReAct agent.
OPENAI_MODEL = 'gpt-4.1-mini'

print("Config loaded.")

Config loaded.


## 5. Connect to the Neo4j Movie Intelligence Graph

We connect using the official Neo4j Python driver. The system requires an **existing** Movie Intelligence
Graph with nodes like `Movie`, `Genre`, `Person`, `Keyword`, `Language` and relationships such as
`(:Movie)-[:HAS_GENRE]->(:Genre)`, `(:Movie)-[:TAGGED_WITH]->(:Keyword)`, `(:Movie)-[:DIRECTED_BY]->(:Person)`,
`(:Person)-[:CAST_IN]->(:Movie)`.

This is the **domain knowledge base** — it is *not* the living memory. Graphiti's living memory (Section 6)
lives in the same Neo4j instance but is namespaced separately via `group_id`. Here we just create the
GraphRAG indexes and confirm the catalogue is populated.


In [4]:
# ---------------------------------------------------------------------------
# Connect to the Neo4j Movie Intelligence Graph (required).
# ---------------------------------------------------------------------------
from neo4j import GraphDatabase
from neo4j_graphrag.indexes import create_vector_index, create_fulltext_index

neo4j_driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
neo4j_driver.verify_connectivity()
print(f"✅ Connected to Neo4j at {NEO4J_URI}")

# Movie-only vector + full-text indexes for GraphRAG hybrid retrieval.
create_vector_index(
    neo4j_driver,
    name="movie_only_embedding_index",
    label="Movie",
    embedding_property="movieEmbedding",
    dimensions=1536,
    similarity_fn="cosine",
)

create_fulltext_index(
    neo4j_driver,
    name="movie_only_text_index",
    label="Movie",
    node_properties=["title", "overview", "tagline"],
)

# Quick sanity check that the Movie Intelligence Graph is populated.
with neo4j_driver.session(database=NEO4J_DATABASE) as s:
    _movie_count = s.run("MATCH (m:Movie) RETURN count(m) AS n").single()["n"]
print(f"🎞️  Movie Intelligence Graph ready: {_movie_count} movies in Neo4j")

✅ Connected to Neo4j at bolt://3.82.224.106
🎞️  Movie Intelligence Graph ready: 496 movies in Neo4j


## 6. The Living Memory Model — **Graphiti as the engine**

Here we initialise the [`graphiti-core`](https://github.com/getzep/graphiti) client and let it *be* the
living-memory layer.

What Graphiti gives us out of the box:

- **`add_episode()`** — ingests a raw user message, uses the LLM to **extract entities & relationships**,
  creates an `Episodic` node, links it via `MENTIONS`, and writes **bitemporal fact edges**
  (`valid_at`, `invalid_at`).
- **Automatic invalidation** — when a new episode contradicts an earlier fact, Graphiti stamps the old
  edge `invalid_at`/`expired_at` instead of deleting it → **full auditable history**.
- **`search()` / `_search()`** — hybrid semantic + BM25 + graph search over the memory, with reranking
  recipes.
- **`group_id` namespacing** — one Neo4j instance holds many users' living graphs alongside the movie KG.

The cell below wires up the OpenAI-backed LLM + embedder and builds Graphiti's indices/constraints.


In [5]:
# ---------------------------------------------------------------------------
# Initialize the Graphiti living-memory client (OpenAI-backed).
# Graphiti does entity/fact extraction, bitemporal tracking, and contradiction
# invalidation for us — we just feed it the user's messages as episodes.
# ---------------------------------------------------------------------------
from graphiti_core import Graphiti
from graphiti_core.llm_client.config import LLMConfig
from graphiti_core.llm_client.openai_client import OpenAIClient
from graphiti_core.embedder.openai import OpenAIEmbedder, OpenAIEmbedderConfig
from graphiti_core.nodes import EpisodeType

# LLM for extraction / dedup / invalidation reasoning.
# IMPORTANT: use OpenAIClient (NOT OpenAIGenericClient). Custom entity/edge
# typing relies on OpenAI structured output, which the *generic* client does
# not handle reliably. With the generic client, typed fact edges (e.g.
# Likes -> sci-fi) silently fail to form and you end up with only bare
# Episodic-[:MENTIONS]->Entity links and no RELATES_TO fact edges.
llm_config = LLMConfig(model=OPENAI_MODEL, small_model=OPENAI_MODEL)
custom_llm = OpenAIClient(config=llm_config)

# Embedder — required so Graphiti can create fact-edge embeddings and search.
embedder = OpenAIEmbedder(
    config=OpenAIEmbedderConfig(embedding_model="text-embedding-3-small")
)

graphiti = Graphiti(
    NEO4J_URI,
    NEO4J_USERNAME,
    NEO4J_PASSWORD,
    llm_client=custom_llm,
    embedder=embedder,
)

# Build Graphiti's own indices & constraints (episodes, entities, edges).
run_async(graphiti.build_indices_and_constraints())
print("✅ Graphiti ready with LLM + Embedder (native living-memory engine).")

✅ Graphiti ready with LLM + Embedder (native living-memory engine).


## 7. Movie-Taste Ontology (Graphiti entity & edge types)

These Pydantic models **are** Graphiti's ontology. They are passed directly to
`graphiti.add_episode(entity_types=..., edge_types=..., edge_type_map=...)`, so Graphiti's LLM extraction
types every entity and relationship it finds.

**Normalization lives in the field descriptions.** Each entity's `canonical_name` description carries the
canonical vocabulary and synonym rules (e.g. *"science fiction"/"scifi" → "sci-fi"*, *"time-travel" →
"time travel"*, *"children getting killed" → "child harm"*). Graphiti's LLM reads these during extraction
and emits already-normalized values that line up with the Movie graph's `Genre`/`Keyword` names — no
separate lookup tables needed. Enum-like fields (`strength`, `sentiment`, `role`, `polarity`) use
`Literal` types so the structured output is constrained and reliable.

**Entity types** (become extra labels on Graphiti's `:Entity` nodes)
- `MovieGenre` — a film genre, snapped to a canonical set (action, horror, sci-fi, comedy …)
- `MovieTheme` — a theme / attribute / sub-style (time travel, heist, slow burn, gore …)
- `MovieTitle` — a specific film the user watched or wants to watch
- `MoviePerson` — a director or actor the user refers to
- `Language` — a spoken language the user prefers (english, korean, hindi …)
- `Era` — a time period / decade (80s, classic, new releases …)
- `ViewingContext` — an occasion or mood (date night, family, solo unwind …)

**Edge types** (stored in the `name` property of Graphiti's `RELATES_TO` edges)
- `Likes` / `Dislikes` — **durable** positive / negative taste toward genres/themes
- `ContentConstraint` — **hard** avoid ("avoid horror", "no gore"), never violated
- `Watched` / `WantsToWatch` — viewing history / watchlist intent
- `Admires` — favored director / actor
- `PrefersLanguage` — preferred (or avoided) spoken language
- `PrefersEra` — preferred decade / period
- `WatchingFor` — the current occasion / mood
- `RequestsNow` — a **momentary, this-turn** ask ("recommend me a thriller"), kept separate from `Likes`
  so a single request doesn't pollute long-term taste

Graphiti makes every edge **bitemporal** (`valid_at`, `invalid_at`, `expired_at`) and **invalidates
contradictions** automatically — e.g. *"I love horror now"* invalidates a prior `Dislikes → horror` while
preserving history. `EDGE_TYPE_MAP` constrains which edge types are allowed between which entity pairs.


In [6]:
# ---------------------------------------------------------------------------
# Custom Graphiti ontology (Pydantic models). Graphiti uses these to type the
# entities/edges it extracts from each episode.
#
# The field DESCRIPTIONS carry the normalization rules that a manual extractor
# would hard-code in lookup tables — the canonical genre vocabulary and the
# "snap synonyms to this form" mapping. Graphiti's LLM reads these descriptions
# during extraction, so it emits already-normalized values (e.g. "science
# fiction"/"scifi" -> "sci-fi") that line up with the Movie graph's Genre names.
# ---------------------------------------------------------------------------
from pydantic import BaseModel, Field
from typing import Optional
from typing import Literal


# Canonical vocabularies (shared in prompts/descriptions so extraction is stable).
CANONICAL_GENRES = (
    "action, adventure, animation, comedy, crime, documentary, drama, fantasy, "
    "horror, mystery, romance, sci-fi, thriller, psychological thriller, war, western"
)
CANONICAL_THEME_EXAMPLES = (
    "time travel, heist, slow burn, mind-bending, dystopian, coming-of-age, "
    "revenge, courtroom, gore, violence, child harm, emotional, feel-good"
)


# ----------------------------- Entity types -------------------------------
class MovieGenre(BaseModel):
    """A movie genre the user refers to."""
    canonical_name: Optional[str] = Field(
        None,
        description=(
            "Genre normalized to lowercase and to ONE of this canonical set: "
            f"{CANONICAL_GENRES}. Snap synonyms to the canonical form: "
            "'science fiction'/'scifi'/'sci fi' -> 'sci-fi'; drop trailing "
            "'movie(s)' ('horror movies' -> 'horror'); "
            "'psychological thriller movie' -> 'psychological thriller'."
        ),
    )


class MovieTheme(BaseModel):
    """A theme, keyword, sub-style, or content attribute of a movie (NOT a genre)."""
    canonical_name: Optional[str] = Field(
        None,
        description=(
            "Theme/keyword/attribute normalized to lowercase singular form, e.g. "
            f"{CANONICAL_THEME_EXAMPLES}. Snap synonyms: 'time-travel' -> 'time "
            "travel'; 'slow-paced'/'slow burn' kept as 'slow burn'; 'mind bending' "
            "-> 'mind-bending'; 'children getting killed'/'child death' -> "
            "'child harm'."
        ),
    )


class MovieTitle(BaseModel):
    """A specific movie the user mentions watching or wanting to watch."""
    year: Optional[int] = Field(None, description="Release year if the user mentions it.")


class MoviePerson(BaseModel):
    """A director or actor the user refers to (e.g. 'Christopher Nolan', 'Tom Hanks')."""
    role: Optional[Literal["director", "actor", "unknown"]] = Field(
        None, description="How the person is referenced."
    )


class Language(BaseModel):
    """A spoken language the user prefers or avoids (e.g. 'english', 'korean', 'hindi')."""
    canonical_name: Optional[str] = Field(
        None, description="Language normalized to lowercase English name (e.g. 'korean')."
    )


class Era(BaseModel):
    """A time period / decade the user prefers (e.g. '1980s', 'classic', 'recent')."""
    canonical_name: Optional[str] = Field(
        None, description="Normalized era label, lowercase (e.g. '90s', 'classic', 'new releases')."
    )


class ViewingContext(BaseModel):
    """An occasion or mood the user is watching for (e.g. 'date night', 'family',
    'solo unwind', 'background watch')."""
    canonical_name: Optional[str] = Field(
        None, description="Normalized context/occasion, lowercase."
    )


# ------------------------------ Edge types --------------------------------
class Likes(BaseModel):
    """User has a DURABLE positive preference toward a genre or theme (a lasting
    taste, e.g. 'I love heist movies'). NOT a one-off 'recommend me X now'."""
    strength: Optional[Literal["mild", "strong", "love"]] = Field(
        None, description="Intensity of the liking."
    )


class Dislikes(BaseModel):
    """User has a DURABLE negative preference toward a genre or theme."""
    strength: Optional[Literal["mild", "strong", "hate"]] = Field(
        None, description="Intensity of the dislike."
    )


class Watched(BaseModel):
    """User has already watched a specific movie."""
    sentiment: Optional[Literal["liked", "disliked", "neutral"]] = Field(
        None, description="How the user felt about it."
    )


class WantsToWatch(BaseModel):
    """User wants to watch / add a specific movie to their watchlist."""


class ContentConstraint(BaseModel):
    """A HARD content constraint the user wants to avoid (e.g. gore, child harm).
    Stronger than Dislikes — must never be violated in a recommendation."""
    reason: Optional[str] = Field(None, description="Why the user wants to avoid it, if stated.")


class Admires(BaseModel):
    """User likes / wants movies from a specific director or actor."""
    role: Optional[Literal["director", "actor"]] = Field(
        None, description="Whether the admired person is a director or actor, if specified."
    )


class PrefersLanguage(BaseModel):
    """User prefers (or avoids) movies in a specific spoken language."""
    polarity: Optional[Literal["prefers", "avoids"]] = Field(
        None, description="Whether the user prefers or avoids this language. Defaults to prefers."
    )


class PrefersEra(BaseModel):
    """User prefers movies from a specific era / decade."""


class WatchingFor(BaseModel):
    """User is choosing a movie for a specific occasion / mood right now."""


class RequestsNow(BaseModel):
    """A MOMENTARY, this-turn request that is NOT a lasting taste — e.g.
    'recommend me a thriller', 'I'm in the mood for something light'. Kept
    separate from Likes so a single ask doesn't pollute long-term preferences."""
    intensity: Optional[Literal["lighter", "neutral", "more intense"]] = Field(
        None, description="Requested tone/intensity for this pick, if stated."
    )


# Register the ontology with Graphiti.
ENTITY_TYPES = {
    "MovieGenre": MovieGenre,
    "MovieTheme": MovieTheme,
    "MovieTitle": MovieTitle,
    "MoviePerson": MoviePerson,
    "Language": Language,
    "Era": Era,
    "ViewingContext": ViewingContext,
}

EDGE_TYPES = {
    "Likes": Likes,
    "Dislikes": Dislikes,
    "Watched": Watched,
    "WantsToWatch": WantsToWatch,
    "ContentConstraint": ContentConstraint,
    "Admires": Admires,
    "PrefersLanguage": PrefersLanguage,
    "PrefersEra": PrefersEra,
    "WatchingFor": WatchingFor,
    "RequestsNow": RequestsNow,
}

# Which edge types are valid between which entity pairs.
# ('Entity', 'Entity') is Graphiti's wildcard fallback for any pair.
EDGE_TYPE_MAP = {
    ("Entity", "MovieGenre"): ["Likes", "Dislikes", "ContentConstraint", "RequestsNow"],
    ("Entity", "MovieTheme"): ["Likes", "Dislikes", "ContentConstraint", "RequestsNow"],
    ("Entity", "MovieTitle"): ["Watched", "WantsToWatch"],
    ("Entity", "MoviePerson"): ["Admires", "Dislikes"],
    ("Entity", "Language"): ["PrefersLanguage"],
    ("Entity", "Era"): ["PrefersEra"],
    ("Entity", "ViewingContext"): ["WatchingFor"],
    ("Entity", "Entity"): ["Likes", "Dislikes", "Watched", "WantsToWatch",
                            "ContentConstraint", "Admires", "PrefersLanguage",
                            "PrefersEra", "WatchingFor", "RequestsNow"],
}

print("✅ Custom Graphiti ontology defined:",
      list(ENTITY_TYPES), "|", list(EDGE_TYPES))

✅ Custom Graphiti ontology defined: ['MovieGenre', 'MovieTheme', 'MovieTitle', 'MoviePerson', 'Language', 'Era', 'ViewingContext'] | ['Likes', 'Dislikes', 'Watched', 'WantsToWatch', 'ContentConstraint', 'Admires', 'PrefersLanguage', 'PrefersEra', 'WatchingFor', 'RequestsNow']


- The **relationship type** is always `RELATES_TO`; the *typed* name (`Likes`, `Dislikes`,
  `ContentConstraint`, `Watched`, `WantsToWatch`, `Admires`, `PrefersLanguage`, `PrefersEra`,
  `WatchingFor`, `RequestsNow`) is stored in the edge's `name` property, driven by our
  `EDGE_TYPES` + `EDGE_TYPE_MAP`.
- Entity **types** (`MovieGenre`, `MovieTheme`, `MovieTitle`, `MoviePerson`, `Language`, `Era`,
  `ViewingContext`) become extra labels on `:Entity`.

In [7]:
# ---------------------------------------------------------------------------
# PURELY GRAPHITI-NATIVE living-memory API.
#
# No manual regex, no manual Cypher writes, no manual invalidation. Everything
# is delegated to Graphiti:
#   • WRITE  -> graphiti.add_episode(..., entity_types, edge_types, edge_type_map)
#              Graphiti's LLM extracts entities/relationships, TYPES them using
#              our Pydantic ontology, writes bitemporal fact edges, and
#              INVALIDATES contradictions automatically.
#   • READ   -> graphiti.search(query, group_ids=[user_id])
#              Hybrid semantic + BM25 + graph search over Graphiti's own facts.
#
# We ACCEPT Graphiti's native storage shape:
#   (:Entity {labels:[..,'MovieGenre']})-[:RELATES_TO {name:'Likes'}]->(:Entity)
# plus Episodic nodes + MENTIONS for provenance — all built by Graphiti.
# ---------------------------------------------------------------------------
from graphiti_core.nodes import EntityNode

# The ontology from Section 7 is what makes Graphiti's extraction typed.
MEMORY_ENTITY_TYPES = ENTITY_TYPES
MEMORY_EDGE_TYPES = EDGE_TYPES
MEMORY_EDGE_TYPE_MAP = EDGE_TYPE_MAP


# ---------------------------------------------------------------------------
# WRITE — one Graphiti call does extraction + typing + invalidation.
# ---------------------------------------------------------------------------
def add_user_message(user_id: str, message: str,
                     reference_time: Optional[datetime] = None) -> Dict[str, Any]:
    """Ingest a user message straight into Graphiti. Graphiti extracts + types
    entities/edges (via our Pydantic ontology), writes bitemporal fact edges,
    links an Episodic provenance node, and invalidates contradictions."""
    now = reference_time or utc_now()
    episode_kwargs = dict(
        name=f"chat-{now.timestamp()}",
        episode_body=f"{user_id}: {message}",
        source=EpisodeType.message,
        source_description="user chat message",
        reference_time=now,
        group_id=user_id,
        entity_types=MEMORY_ENTITY_TYPES,
        edge_types=MEMORY_EDGE_TYPES,
        edge_type_map=MEMORY_EDGE_TYPE_MAP,
    )
    # Prefer incremental community refresh, but never let it break the write.
    # Graphiti's incremental community path only works cleanly once a community
    # structure already exists. On cold start (no communities yet) it raises a
    # BENIGN error — seen as either a "too many values to unpack (expected 2)"
    # ValueError OR a pydantic ValidationError complaining that `communities` /
    # `community_edges` are empty lists (CommunityNode/CommunityEdge expected).
    # In both cases we simply retry the SAME write without the incremental
    # refresh so the typed fact edges still commit. Full clustering is always
    # available via build_taste_communities() ("Rebuild taste clusters"), so
    # this is expected and silent. Any OTHER (genuinely unexpected) error is
    # surfaced.
    try:
        result = run_async(graphiti.add_episode(**episode_kwargs,
                                                update_communities=True))
    except Exception as e:
        _msg = str(e).lower()
        _expected_cold_start = (
            (isinstance(e, ValueError) and "unpack" in _msg)
            or "community" in _msg          # CommunityNode / community_edges / communities
            or "communitynode" in _msg
        )
        if not _expected_cold_start:
            print("add_episode note: incremental community update skipped "
                  f"({type(e).__name__}: {e}). Committing facts without it.")
        result = run_async(graphiti.add_episode(**episode_kwargs))

    edges = getattr(result, "edges", []) or []
    nodes = getattr(result, "nodes", []) or []
    episode_uuid = getattr(getattr(result, "episode", None), "uuid", None)
    return {
        "updated": bool(edges),
        "episode_uuid": episode_uuid,
        "facts": [{"edge_type": e.name, "fact": e.fact} for e in edges],
        "entities": [n.name for n in nodes],
        "summary": f"Graphiti committed {len(edges)} typed fact edge(s).",
    }


# ---------------------------------------------------------------------------
# READ — Graphiti hybrid search over its own fact edges.
# ---------------------------------------------------------------------------
def search_user_memory(user_id: str, query: str = "",
                       current_only: bool = True) -> Dict[str, Any]:
    """Read the user's memory via graphiti.search(). Returns the relevant,
    currently-valid fact edges (Graphiti excludes invalidated ones by default).
    `current_only` is kept for signature compatibility — Graphiti already
    returns the live bitemporal state."""
    q = query or "the user's movie preferences: likes, dislikes, avoids, watchlist"
    try:
        edges = run_async(graphiti.search(query=q, group_ids=[user_id], num_results=25))
    except Exception as e:
        # Fact-edge search never touches communities, so it cannot raise the
        # community_edges validation error — but guard anyway so a transient /
        # cold-start hiccup can't crash a chat turn or the preference panel.
        print(f"search_user_memory note: graphiti.search skipped ({type(e).__name__}: {e}).")
        edges = []

    facts = [e.fact for e in edges]
    return {
        "active_preferences": facts,
        "preference_summary": "; ".join(facts) if facts
            else "No known preferences yet (cold start).",
        "rows": [{
            "edge_type": e.name,
            "fact": e.fact,
            "valid_at": getattr(e, "valid_at", None),
            "invalid_at": getattr(e, "invalid_at", None),
        } for e in edges],
    }


def inspect_user_facts(user_id: str, current_only: bool = True) -> pd.DataFrame:
    """Audit view of Graphiti's fact edges (via graphiti.search) as a DataFrame."""
    edges = run_async(graphiti.search(
        query="the user's movie preferences and viewing history",
        group_ids=[user_id], num_results=50))
    rows = [{
        "edge_type": e.name,
        "fact": e.fact,
        "valid_at": getattr(e, "valid_at", None),
        "invalid_at": getattr(e, "invalid_at", None),
    } for e in edges]
    return pd.DataFrame(rows)


# ---------------------------------------------------------------------------
# HISTORY-AWARE READ — for TIMELINE / "how have my tastes changed" questions.
#
# graphiti.search() only returns LIVE (non-invalidated) edges, so it can't feed
# a "Then -> Now" timeline. Here we read Graphiti's bitemporal store DIRECTLY,
# INCLUDING superseded/invalidated edges (invalid_at / expired_at set), ordered
# oldest -> newest so the agent (or the UI button) can reconstruct the real
# evolution of the user's preferences with dates.
# ---------------------------------------------------------------------------
def search_user_memory_history(user_id: str, query: str = "") -> Dict[str, Any]:
    """Return the FULL bitemporal history of a user's typed fact edges — active
    AND invalidated — straight from Graphiti's RELATES_TO edges, oldest first.
    `query` is accepted for tool-signature compatibility (history is returned
    whole; the agent filters/among it as needed)."""
    cypher = """
    MATCH (a:Entity)-[r:RELATES_TO]->(b:Entity)
    WHERE r.group_id = $uid AND r.fact IS NOT NULL
    RETURN r.name        AS edge_type,
           r.fact        AS fact,
           a.name        AS source,
           b.name        AS target,
           r.valid_at    AS valid_at,
           r.invalid_at  AS invalid_at,
           r.expired_at  AS expired_at
    ORDER BY coalesce(r.valid_at, r.created_at)
    """
    try:
        with neo4j_driver.session(database=NEO4J_DATABASE) as s:
            rows = [dict(r) for r in s.run(cypher, uid=user_id)]
    except Exception as e:
        print(f"search_user_memory_history note: read skipped ({type(e).__name__}: {e}).")
        rows = []

    for r in rows:
        # A fact is still active if it has neither invalid_at nor expired_at.
        r["status"] = "active" if not (r.get("invalid_at") or r.get("expired_at")) else "superseded"

    active = [r for r in rows if r["status"] == "active"]
    superseded = [r for r in rows if r["status"] == "superseded"]
    return {
        "rows": rows,
        "active": active,
        "superseded": superseded,
        "has_history": bool(superseded),
        "summary": f"{len(rows)} total fact(s): {len(active)} active, "
                   f"{len(superseded)} superseded/invalidated.",
    }


def taste_timeline_df(user_id: str) -> pd.DataFrame:
    """Bitemporal history (active + invalidated) as a tidy DataFrame for the UI."""
    hist = search_user_memory_history(user_id)
    return pd.DataFrame(hist["rows"])


print("✅ Graphiti-native memory API ready: add_user_message / search_user_memory / inspect_user_facts / search_user_memory_history")


✅ Graphiti-native memory API ready: add_user_message / search_user_memory / inspect_user_facts / search_user_memory_history


## 9. Episodic Memory & Provenance — from Graphiti

Every `add_episode()` call stores an `Episodic` node and links it to the entities it mentions via
`MENTIONS` edges — recording *where each belief came from*, exactly as described in the Graphiti docs.

Below we retrieve episodes and their mentioned entities directly with a tiny helper that reads Graphiti's
own schema.


In [8]:
# ---------------------------------------------------------------------------
# Episode / provenance helpers — read Graphiti's OWN Episodic + MENTIONS schema.
# (Graphiti creates these automatically inside add_episode; we only READ them.)
# ---------------------------------------------------------------------------
def inspect_episodes(user_id: str) -> pd.DataFrame:
    cypher = """
    MATCH (e:Episodic {group_id: $uid})
    OPTIONAL MATCH (e)-[:MENTIONS]->(n)
    RETURN e.uuid        AS episode_uuid,
           e.name        AS episode_name,
           e.content     AS content,
           e.created_at  AS created_at,
           collect(DISTINCT n.name) AS mentioned_entities
    ORDER BY e.created_at DESC
    """
    with neo4j_driver.session(database=NEO4J_DATABASE) as s:
        rows = [dict(r) for r in s.run(cypher, uid=user_id)]
    return pd.DataFrame(rows)


# ---------------------------------------------------------------------------
# Smoke test — watch Graphiti extract, type, and INVALIDATE facts on its own.
# (Uncomment to run; uses a throwaway user namespace.)
# ---------------------------------------------------------------------------
# demo_uid = "user_demo"
# print(add_user_message(demo_uid, "I love horror movies and heist thrillers"))
# print(add_user_message(demo_uid, "Actually I hate horror now, it's not for me"))
# print("\nCurrent memory:", search_user_memory(demo_uid, "what does the user like?")["preference_summary"])
# display(inspect_user_facts(demo_uid, current_only=False))  # full bitemporal history
# display(inspect_episodes(demo_uid))

print("✅ Episode/provenance helper ready (reads Graphiti's native Episodic + MENTIONS).")

✅ Episode/provenance helper ready (reads Graphiti's native Episodic + MENTIONS).


## 10. Communities — high-level taste clusters (native Graphiti)

Graphiti's **`build_communities()`** runs the Leiden algorithm to group strongly-connected entity nodes
into `CommunityNode`s, each with an LLM-synthesized summary. For a movie agent this surfaces **emergent
taste clusters** — e.g. *"dark psychological sci-fi"* vs *"feel-good comedies"* — a high-level *"who is
this viewer"* picture rather than only scattered `Likes`/`Dislikes` edges.

These clusters are **actively used** downstream:

- **`taste_profile` agent tool** (Section 14) — on vague requests (*"surprise me"*, *"what should I watch
  tonight?"*) the ReAct agent consults the cluster summary to understand the whole viewer before
  recommending.
- **`movie_hybrid_retriever`** (Section 11) — blends the top cluster summary into the hybrid-search query
  text, biasing recall toward the user's dominant themes.

`add_user_message` passes **`update_communities=True`**, so clusters refresh incrementally as new facts
arrive; the chat UI also has a **Rebuild taste clusters** button for a full Leiden re-run on demand.



In [9]:
# ---------------------------------------------------------------------------
# Communities — native Graphiti Leiden clustering + LLM summaries.
#
# Communities group a user's strongly-connected taste entities (genres, themes,
# people, languages, eras) into CommunityNodes, each with an LLM-synthesized
# summary. That gives us a HIGH-LEVEL "who is this viewer" picture — e.g.
# "leans cerebral, slow-burn sci-fi + heist thrillers; avoids gore" — instead
# of only scattered individual Likes/Dislikes edges.
#
# We use them in two places downstream:
#   • taste_profile tool  -> the ReAct agent consults the cluster summary on
#                            vague/open-ended requests ("surprise me").
#   • movie_hybrid_retriever -> blends the top cluster summary into the search
#                            text so hybrid recall reflects the WHOLE taste.
# ---------------------------------------------------------------------------
from graphiti_core.search.search_config_recipes import COMMUNITY_HYBRID_SEARCH_RRF


def build_taste_communities():
    """Cluster the living-memory entities into communities with summaries.

    Returns True on success, False if Graphiti's community build raised (e.g.
    on a tiny/degenerate graph). NEVER propagates — a failed cluster build must
    not crash a chat turn or the demo seed. Full clustering can simply be
    retried later once more preferences exist."""
    try:
        run_async(graphiti.build_communities())
        print("✅ Communities built (Leiden clustering + LLM summaries).")
        return True
    except Exception as e:
        print("⚠️ Community build skipped this time "
              f"({type(e).__name__}: {e}). "
              "This is safe — try again after adding a few more preferences.")
        return False


def search_communities(user_id: str, query: str, limit: int = 3) -> List[Dict[str, str]]:
    """Retrieve high-level taste clusters relevant to a query. Returns a list of
    {name, summary}. Safe on cold start (returns [] if no communities exist)."""
    cfg = COMMUNITY_HYBRID_SEARCH_RRF.model_copy(deep=True)
    cfg.limit = limit
    try:
        res = run_async(graphiti._search(
            query=query,
            group_ids=[user_id],
            config=cfg,
        ))
    except Exception:
        return []
    return [
        {"name": c.name, "summary": getattr(c, "summary", "") or ""}
        for c in getattr(res, "communities", [])
    ]


def taste_profile_for_user(user_id: str,
                           query: str = "overall movie taste and viewing habits",
                           limit: int = 3) -> str:
    """Compact, human-readable taste profile built from Graphiti communities.
    Used by the taste_profile tool AND blended into hybrid retrieval. Returns
    an empty string if no communities have been built yet (cold start)."""
    clusters = search_communities(user_id, query, limit=limit)
    parts = []
    for c in clusters:
        summary = c.get("summary", "").strip()
        name = c.get("name", "").strip()
        if summary:
            parts.append(f"{name}: {summary}" if name else summary)
        elif name:
            parts.append(name)
    return " | ".join(parts).strip()


# Example (run after a user has some history):
# build_taste_communities()
# print(taste_profile_for_user("user_harika"))
print("✅ Community helpers ready: build_taste_communities / search_communities / taste_profile_for_user")


✅ Community helpers ready: build_taste_communities / search_communities / taste_profile_for_user


### 10a. Profile the Movie graph & seed a dataset-aligned demo taste

The communities demo only looks good if the seeded preferences match what the **Movie Intelligence Graph
actually contains**. The first cell below profiles the catalogue (best-represented genres, keywords,
directors, cast, languages) using the already-authenticated `neo4j_driver`. The second builds a
**dataset-aligned seed profile** for `CHAT_USER` from those real values, then runs a full
`build_taste_communities()` so the 👥 taste clusters and `taste_profile` tool light up with retrievable
themes — no hallucinated genres.


In [10]:
# ---------------------------------------------------------------------------
# Profile the Movie Intelligence Graph so the demo seeds align with what the
# catalogue can actually retrieve. Uses the already-authenticated neo4j_driver.
# ---------------------------------------------------------------------------
def profile_movie_graph(top_n: int = 15) -> Dict[str, List[tuple]]:
    """Return the best-represented genres / keywords / directors / cast /
    languages in the Movie graph (name, count), most frequent first."""
    queries = {
        "genres": """
            MATCH (m:Movie)-[:HAS_GENRE]->(g:Genre)
            RETURN g.name AS name, count(m) AS n ORDER BY n DESC LIMIT $k""",
        "keywords": """
            MATCH (m:Movie)-[:TAGGED_WITH]->(x:Keyword)
            RETURN x.name AS name, count(m) AS n ORDER BY n DESC LIMIT $k""",
        "directors": """
            MATCH (m:Movie)-[:DIRECTED_BY]->(p:Person)
            RETURN p.name AS name, count(m) AS n ORDER BY n DESC LIMIT $k""",
        "cast": """
            MATCH (p:Person)-[:CAST_IN]->(m:Movie)
            RETURN p.name AS name, count(m) AS n ORDER BY n DESC LIMIT $k""",
        "languages": """
            MATCH (m:Movie)-[:SPOKEN_IN]->(l:Language)
            RETURN l.name AS name, count(m) AS n ORDER BY n DESC LIMIT $k""",
    }
    out: Dict[str, List[tuple]] = {}
    with neo4j_driver.session(database=NEO4J_DATABASE) as s:
        for key, cy in queries.items():
            try:
                out[key] = [(r["name"], r["n"]) for r in s.run(cy, k=top_n)]
            except Exception as e:
                out[key] = []
                print(f"  (skip {key}: {e})")
    return out


MOVIE_GRAPH_PROFILE = profile_movie_graph(top_n=15)

for _section, _rows in MOVIE_GRAPH_PROFILE.items():
    print(f"\nTOP {_section.upper()}:")
    for _name, _cnt in _rows:
        print(f"  {str(_name):<30} {_cnt}")



TOP GENRES:
  Drama                          266
  Action                         191
  Thriller                       170
  Comedy                         152
  Romance                        117
  Crime                          95
  Adventure                      40
  Mystery                        33
  Horror                         30
  History                        26
  Family                         23
  Fantasy                        17
  Science Fiction                13
  War                            9
  Documentary                    8

TOP KEYWORDS:
  india                          22
  based on true story            21
  biography                      21
  based on novel or book         17
  revenge                        14
  remake                         14
  sports                         13
  police                         11
  sequel                         10
  love                           10
  based on movie                 10
  serial killer                  

In [11]:
# ---------------------------------------------------------------------------
# Seed a DATASET-ALIGNED demo taste for the communities showcase.
#
# Instead of hard-coding "cerebral sci-fi" (which this catalogue may not hold),
# we build the seed messages from the REAL top genres / keywords / directors
# returned by profile_movie_graph(). That guarantees the seeded Likes/Admires
# edges have retrievable movies behind them, so communities cluster around
# themes the hybrid retriever can actually surface — no hallucinated genres.
# ---------------------------------------------------------------------------
def seed_demo_taste_from_dataset(user_id: str,
                                 profile: Dict[str, List[tuple]] = None,
                                 n_genres: int = 3, n_keywords: int = 3,
                                 n_directors: int = 1,
                                 rebuild_communities: bool = True) -> None:
    profile = profile or MOVIE_GRAPH_PROFILE
    top = lambda key, n: [name for name, _ in (profile.get(key) or [])[:n] if name]

    genres = top("genres", n_genres)
    keywords = top("keywords", n_keywords)
    directors = top("directors", n_directors)

    # Natural-language preference messages -> Graphiti extracts + types them.
    messages = []
    if genres:
        messages.append(f"I really love {', '.join(genres)} movies.")
    if keywords:
        messages.append(f"I'm especially drawn to films about {', '.join(keywords)}.")
    for d in directors:
        messages.append(f"{d} is one of my favorite directors.")

    if not messages:
        print("⚠️ No dataset profile available — run the profiler cell first.")
        return

    print(f"Seeding {len(messages)} dataset-aligned preference(s) for '{user_id}':")
    for msg in messages:
        print("  •", msg)
        res = add_user_message(user_id, msg)
        print("    →", res["summary"])

    if rebuild_communities:
        print("\nBuilding taste communities (full Leiden pass)…")
        built = build_taste_communities()
        if built:
            prof = taste_profile_for_user(user_id)
            print("👥 taste_profile:", prof or "(no clusters formed yet)")
        else:
            print("👥 taste_profile: (clusters not built yet — the facts are "
                  "committed; click 'Rebuild taste clusters' or re-run after "
                  "adding more preferences)")


# Run it for the chat user to prime the communities demo:
# seed_demo_taste_from_dataset(CHAT_USER)
print("✅ Seed helper ready: seed_demo_taste_from_dataset(CHAT_USER)")


✅ Seed helper ready: seed_demo_taste_from_dataset(CHAT_USER)


## 11. Neo4j GraphRAG Hybrid Movie Retriever Tool

This tool wraps Neo4j GraphRAG's `HybridRetriever` (vector + full-text, fused by RRF) over the Movie
Intelligence Graph, then graph-expands each candidate.

It reads the user's preferences from **Graphiti's living memory** (`graphiti_signals_for_user`) and maps
the active fact edges into positive signals (likes) and hard filters (dislikes / content constraints).

Each candidate returns: `title`, `genres`, `keywords`, `directors`, `cast`, `overview`, `rating`,
`why_retrieved`, `matched_preferences`, `possible_risks`.


In [12]:
# ---------------------------------------------------------------------------
# Graph expansion helper: enrich candidate titles with genres/keywords/cast.
# ---------------------------------------------------------------------------
def enrich_movies(driver, titles):
    query = """
    MATCH (m:Movie)
    WHERE m.title IN $titles
    OPTIONAL MATCH (m)-[:HAS_GENRE]->(g:Genre)
    OPTIONAL MATCH (m)-[:TAGGED_WITH]->(k:Keyword)
    OPTIONAL MATCH (m)-[:DIRECTED_BY]->(d:Person)
    OPTIONAL MATCH (p:Person)-[:CAST_IN]->(m)
    OPTIONAL MATCH (m)-[:PRODUCED_BY]->(pc:ProductionCompany)
    OPTIONAL MATCH (m)-[:PRODUCED_IN]->(c:Country)
    OPTIONAL MATCH (m)-[:SPOKEN_IN]->(l:Language)
    RETURN
        m.title AS title,
        m.overview AS overview,
        m.vote_average AS rating,
        // Release year — coalesce across the property names commonly seen on
        // Movie nodes (release_year / year / a 'YYYY-...' release_date string).
        coalesce(
            m.release_year,
            m.year,
            CASE WHEN m.release_date IS NULL THEN NULL
                 ELSE toInteger(left(toString(m.release_date), 4)) END
        ) AS release_year,
        collect(DISTINCT g.name) AS genres,
        collect(DISTINCT k.name) AS keywords,
        collect(DISTINCT d.name) AS directors,
        collect(DISTINCT p.name) AS cast,
        collect(DISTINCT pc.name) AS production_companies,
        collect(DISTINCT c.name) AS countries,
        collect(DISTINCT l.name) AS languages
    """
    return driver.execute_query(query, titles=titles, database_=NEO4J_DATABASE).records

In [13]:
# ---------------------------------------------------------------------------
# Turn the user's ACTIVE Graphiti fact edges into retrieval signals.
# Reads Graphiti's NATIVE storage shape:
#   (:Entity)-[:RELATES_TO {name:'Likes'|'Dislikes'|'ContentConstraint'|
#                                 'Admires'|'PrefersLanguage'|'PrefersEra'}]->(:Entity)
# keeping only edges that are not invalidated/expired (live bitemporal state).
#
# Edge-name -> signal bucket (matches the Pydantic edge types in Section 7):
#   Likes            -> likes
#   Dislikes         -> dislikes
#   ContentConstraint-> constraints (hard avoid)
#   Admires          -> people   (favored director/actor)
#   PrefersLanguage  -> languages (favored spoken language)
#   PrefersEra       -> eras     (favored decade / period)
# ---------------------------------------------------------------------------
def graphiti_signals_for_user(user_id: str) -> Dict[str, List[str]]:
    cypher = """
    MATCH (a:Entity)-[r:RELATES_TO]->(b:Entity)
    WHERE r.group_id = $uid
      AND r.invalid_at IS NULL AND r.expired_at IS NULL
      AND r.name IN ['Likes','Dislikes','ContentConstraint',
                     'Admires','PrefersLanguage','PrefersEra']
    RETURN r.name AS edge_type, b.name AS target, properties(r) AS props
    """
    with neo4j_driver.session(database=NEO4J_DATABASE) as s:
        rows = [dict(r) for r in s.run(cypher, uid=user_id)]

    likes, dislikes, constraints = [], [], []
    people, languages, eras = [], [], []
    for r in rows:
        et = (r.get("edge_type") or "")
        target = (r.get("target") or "").lower().strip()
        props = r.get("props") or {}
        if not target:
            continue
        if et == "Likes":
            likes.append(target)
        elif et == "Dislikes":
            dislikes.append(target)
        elif et == "ContentConstraint":
            constraints.append(target)
        elif et == "Admires":
            people.append(target)
        elif et == "PrefersLanguage":
            # treat as positive unless the user explicitly avoids the language
            if (props.get("polarity") or "prefers").lower() != "avoids":
                languages.append(target)
        elif et == "PrefersEra":
            eras.append(target)
    return {
        "likes": sorted(set(likes)),
        "dislikes": sorted(set(dislikes)),
        "constraints": sorted(set(constraints)),
        "people": sorted(set(people)),
        "languages": sorted(set(languages)),
        "eras": sorted(set(eras)),
    }


# Map content constraints to attribute red-flags for filtering.
CONSTRAINT_RISK = {
    "child harm": ["violent", "gory", "dark"],
    "gore": ["gory"],
    "violence": ["violent", "gory"],
    "horror": ["gory", "violent", "dark"],
}


# ---------------------------------------------------------------------------
# Era -> (min_year, max_year) parser. Turns a stored Era label like "80s",
# "1990s", "classic", or "recent" into a release-year window for filtering.
# Returns None when the label can't be interpreted (so it won't filter).
# ---------------------------------------------------------------------------
import re
from datetime import date

def era_to_year_range(era: str):
    e = (era or "").lower().strip()
    if not e:
        return None
    this_year = date.today().year

    # loose qualitative labels
    if any(w in e for w in ["recent", "new release", "new releases", "latest", "modern"]):
        return (this_year - 7, this_year)
    if any(w in e for w in ["classic", "old", "golden age", "vintage"]):
        return (1900, 1979)

    # explicit decade like "80s", "1980s", "'90s"
    m = re.search(r"(19|20)?(\d0)s", e)
    if m:
        century = m.group(1) or ("19" if int(m.group(2)) >= 30 else "20")
        start = int(century + m.group(2))
        return (start, start + 9)

    # explicit 4-digit year -> that single year
    m = re.search(r"(19|20)\d{2}", e)
    if m:
        y = int(m.group(0))
        return (y, y)
    return None


# ---------------------------------------------------------------------------
# Safe numeric coercion for values coming back from Neo4j. Movie nodes can
# store release_year / vote_average as ints, floats, OR strings (e.g. "7.8",
# "1994"). Mixing str and float in comparisons/sorts raises
# "'<' not supported between instances of 'str' and 'float'". This normalizes
# any of those to a float (or None when it isn't a real number).
# ---------------------------------------------------------------------------
def _as_number(value):
    if value is None:
        return None
    if isinstance(value, (int, float)):
        return float(value)
    try:
        return float(str(value).strip())
    except (TypeError, ValueError):
        return None


# ---------------------------------------------------------------------------
# Neo4j GraphRAG hybrid retriever (vector index + full-text index).
# ---------------------------------------------------------------------------
import ast
from langchain_core.tools import tool
from neo4j_graphrag.retrievers import HybridRetriever
from neo4j_graphrag.embeddings import OpenAIEmbeddings

_embedder = OpenAIEmbeddings(model="text-embedding-3-small")

_hybrid_retriever = HybridRetriever(
    driver=neo4j_driver,
    vector_index_name="movie_only_embedding_index",
    fulltext_index_name="movie_only_text_index",
    embedder=_embedder,
)


def _graphrag_retrieve(query_text: str, top_k: int) -> List[Dict[str, Any]]:
    res = _hybrid_retriever.search(query_text=query_text, top_k=top_k)
    out = []
    for item in res.items:
        rec = {}
        if item.content:
            try:
                rec.update(ast.literal_eval(item.content))
            except Exception:
                pass
        rec.update(item.metadata or {})
        out.append(rec)
    return out


@tool
def movie_hybrid_retriever(user_id: str, recommendation_query: str = "",
                           top_k: int = 4) -> List[Dict[str, Any]]:
    """Retrieve candidate movies from the Neo4j Movie Intelligence Graph via
    GraphRAG hybrid vector + graph retrieval, honoring the user's living-memory
    likes / admired people / preferred language / preferred era (positive
    signals) and dislikes / avoids (hard filters). Never returns movies
    violating active dislikes/constraints.
    """
    signals = graphiti_signals_for_user(user_id)

    # High-level taste cluster (Graphiti communities). Biases hybrid recall
    # toward the user's DOMINANT themes, not just literal Likes tokens. Empty
    # on cold start, so it only helps once some history exists.
    profile = taste_profile_for_user(user_id, recommendation_query or
                                      "overall movie taste")

    # Blend positive signals (likes + admired people + taste profile) into the
    # search text.
    query_parts = [recommendation_query] + signals["likes"] + signals["people"]
    if profile:
        query_parts.append(profile)
    query_text = " ".join(p for p in query_parts if p).strip() or "movie"
    raw = _graphrag_retrieve(query_text, top_k * 4)

    candidate_titles = [m["title"] for m in raw if m.get("title")]
    movies = enrich_movies(neo4j_driver, candidate_titles)

    dislikes = set(signals["dislikes"])
    pref_langs = set(signals["languages"])
    fav_people = set(signals["people"])

    # Turn the user's era preferences into year windows once.
    era_ranges = [r for r in (era_to_year_range(e) for e in signals["eras"]) if r]

    candidates = []
    for m in movies:
        genres = {g.lower() for g in m.get("genres", [])}
        themes = {k.lower() for k in m.get("keywords", [])}
        langs = {l.lower() for l in m.get("languages", [])}
        crew = {p.lower() for p in m.get("directors", []) + m.get("cast", [])}
        attrs = set(themes)
        # Coerce numeric fields defensively: Neo4j Movie nodes sometimes store
        # release_year / vote_average as strings, which would otherwise blow up
        # the year comparison and the final sort ("'<' not supported between
        # instances of 'str' and 'float'").
        year = _as_number(m.get("release_year"))
        rating = _as_number(m.get("rating")) or 0.0

        # ---- hard filter: dislikes ----
        if (genres & dislikes) or (themes & dislikes):
            continue

        # ---- hard filter: avoids / content constraints ----
        risks = []
        for c in signals["constraints"]:
            # direct match (e.g. avoid "horror" genre) or attribute red-flag
            if c in genres or c in themes:
                risks.append(f"avoid:{c}")
            flagged = attrs & set(CONSTRAINT_RISK.get(c, []))
            if flagged:
                risks.append(f"{c}: flagged by {sorted(flagged)}")
        if risks:
            continue

        # ---- soft filter: preferred language (skip only if the user has a
        #      language preference AND this movie matches none of them) ----
        if pref_langs and not (langs & pref_langs):
            continue

        # ---- soft filter: preferred era (skip only if the user has an era
        #      preference, the movie has a known year, and it's outside every
        #      preferred window) ----
        era_match = False
        if era_ranges:
            if year is not None:
                era_match = any(lo <= year <= hi for lo, hi in era_ranges)
                if not era_match:
                    continue
            # unknown year -> don't filter out, just can't confirm the match

        likes = set(signals["likes"])
        matched = [f"genre:{g}" for g in genres & likes] + \
                  [f"keyword:{k}" for k in themes & likes] + \
                  [f"person:{p}" for p in crew & fav_people] + \
                  [f"language:{l}" for l in langs & pref_langs]
        if era_match and year is not None:
            matched.append(f"era:{year}")

        candidates.append({
            "title": m["title"],
            "overview": m.get("overview", ""),
            "rating": rating,
            "release_year": year,
            "genres": m.get("genres", []),
            "keywords": m.get("keywords", []),
            "directors": m.get("directors", []),
            "cast": m.get("cast", []),
            "languages": m.get("languages", []),
            "matched_preferences": matched,
            "why_retrieved": f"Matched GraphRAG search for '{query_text}'",
            "possible_risks": risks,
        })

    # Rank: more matched preferences first, then rating. Both keys are numeric
    # (rating is coerced above), so the sort can't hit str-vs-float errors.
    candidates.sort(key=lambda c: (len(c["matched_preferences"]), c["rating"] or 0.0),
                    reverse=True)
    return candidates[:top_k]


print("✅ Tool defined: movie_hybrid_retriever (GraphRAG + Graphiti-native signals)")

✅ Tool defined: movie_hybrid_retriever (GraphRAG + Graphiti-native signals)


## 12. Web Search Tool: `web_search`

Used **only** when the graph lacks enough metadata or a candidate needs validation (e.g. *is this
slow-paced? violent? recent?*). Backed by **Tavily** — set `WEB_SEARCH_API_KEY` to your Tavily key.


In [14]:
# ---------------------------------------------------------------------------
# web_search — validates external / missing movie attributes. Uses Tavily.
# ---------------------------------------------------------------------------
from tavily import TavilyClient

_tavily = TavilyClient(api_key=WEB_SEARCH_API_KEY)


@tool
def web_search(query: str, movie_title: str = "") -> Dict[str, Any]:
    """Validate external / missing movie attributes (pace, violence, freshness,
    popularity, reviews). Use ONLY when the graph lacks enough metadata.
    Returns a short direct answer to minimize token usage.
    """
    q = f"{movie_title} {query}".strip()
    res = _tavily.search(query=q, max_results=1, include_answer=True)

    answer = (res.get("answer") or "").strip()
    if answer:
        return {"external_summary": answer, "source": "tavily"}

    results = res.get("results", [])
    if results:
        top = results[0]
        return {"external_summary": top.get("content", "")[:150],
                "source": top.get("url", "tavily")}
    return {"external_summary": "No information found.", "source": "tavily"}


print("✅ Tool defined: web_search (Tavily)")

✅ Tool defined: web_search (Tavily)


## 13. ReAct Agent Prompt (tool policy)

We use a **LangChain ReAct agent** — the LLM decides, turn by turn, which tool to call. The prompt below
describes each tool and enforces the **memory-first workflow**: search memory first, and call
`update_memory` (→ `graphiti.add_episode`) the moment the user states or changes a preference, so the
living graph is always up to date before recommending.


In [15]:
# ---------------------------------------------------------------------------
# ReAct agent instructions (injected into the ReAct prompt template).
# ---------------------------------------------------------------------------
AGENT_INSTRUCTIONS = """You are a personalized movie recommendation assistant powered by a \
Graphiti LIVING MEMORY GRAPH. You learn the user's tastes over the conversation and keep their \
preferences continuously up to date.


You have access to these tools:


- search_memory: Look up the user's CURRENT active preferences from the Graphiti living memory graph. \
ALWAYS call this FIRST on every turn, before doing anything else.


- update_memory: Record or update the user's preferences by adding the message as a Graphiti episode. \
Call this IMMEDIATELY every time the user states, changes, narrows, or contradicts a preference. Graphiti \
extracts the facts, types them, and invalidates contradicting old ones automatically (history is \
preserved). Do NOT skip this step when a preference appears.


- taste_profile: Get a HIGH-LEVEL summary of the user's overall taste, distilled from Graphiti \
communities (clusters of their genres/themes/people). Use this on VAGUE or open-ended requests \
("surprise me", "what should I watch tonight", "pick something good") to understand the whole viewer \
before recommending. It may be empty for brand-new users — then just rely on search_memory.


- taste_timeline: Retrieve the user's FULL preference HISTORY from the Graphiti bitemporal store, \
INCLUDING superseded / invalidated facts (with valid_at / invalid_at stamps). Call this ONLY when the \
user asks how their tastes have EVOLVED or CHANGED over time (\"how have my preferences changed?\", \
\"show my taste timeline\", \"what did I used to like?\"). Do NOT use it for normal recommendations — \
search_memory already returns the current active preferences. Render its output per the TIMELINE section.


- movie_hybrid_retriever: Retrieve candidate movies from the Neo4j Movie Intelligence Graph using hybrid \
vector + graph retrieval. Use this when the user asks for a recommendation. Always run search_memory (and \
any needed update_memory) BEFORE calling it.


- web_search: Validate missing or uncertain movie attributes (pace, violence, tone, freshness, rating). \
Use only when retrieval metadata is incomplete or you need external confirmation.


TOOL POLICY (follow every turn):
1. Call search_memory first.
2. If the user's message contains ANY new, changed, narrowing, or contradictory preference, call \
update_memory immediately with their ORIGINAL words. This includes:
   - New tastes ("I like sci-fi thrillers").
   - Contradictions ("actually I avoid horror" after liking it) — Graphiti invalidates the old fact and \
keeps the newer one live; the old one stays in history.
   - Narrowing refinements ("psychological horror is okay if not gory", "okay with slow movies if very \
tense") — these REPLACE a broad preference with a more specific one. Treat them as an update, not noise.
   - Hard constraints ("no gore", "avoid horror") — store as a ContentConstraint.
3. If the user is only sharing info (not asking for a movie), acknowledge naturally — do NOT force a rec.
4. If the user asks for a recommendation:
   a. Treat what they explicitly asked for THIS turn as the main signal.
   b. Use stored preferences to disambiguate, honor hard dislikes/constraints, and personalize ranking.
   c. Do NOT silently inject stored preference words into the current request.
   d. After retrieval, reject any candidate conflicting with an active dislike or ContentConstraint \
(e.g. never return a gory movie when "no gore" is active, even if it matches the genre).
   e. Use web_search only to verify uncertain metadata (pace, gore, tone).
   f. Explain briefly why each recommendation fits.
5. If the recommendation request is VAGUE or open-ended ("surprise me", "what should I watch tonight", \
"pick something good", "I don't know, you choose"), treat it as DISCOVERY / SURPRISE MODE: call \
taste_profile AFTER search_memory to pull the user's high-level taste cluster, use it to steer \
movie_hybrid_retriever, AND ALSO call web_search to blend in fresh, external discovery (buzzworthy / \
trending / critically-loved picks aligned to that taste). Combine both sources in your answer per the \
DISCOVERY MODE rules below. Skip this blend when the user already gave a concrete, specific ask (e.g. "a \
recent Korean heist thriller") — then rely on the catalogue and only verify with web_search if needed.

CLARIFICATION LIMIT (strict — this prevents question loops):
- You may ask AT MOST ONE clarifying question across an ENTIRE recommendation attempt, and only right \
after the first recommendation request when you genuinely cannot proceed.
- Treat vague refinement phrases as ANSWERS, not triggers for new questions. Phrases like "something \
less intense", "something lighter", "keep it light", "just less intense", "less intense thriller", \
"something slower but tense" are VALID narrowing signals — apply them directly and recommend.
- NEVER ask a second clarifying question, and NEVER re-ask a reworded variant of a question you already \
asked. If you already asked once, you MUST proceed to search_memory → movie_hybrid_retriever and \
recommend using your best interpretation.
- When in doubt, prefer recommending (with a one-line note of your assumption) over asking again.

INTENSITY / TONE MAPPING (apply narrowing signals yourself, don't ask):
- "less intense" / "lighter" / "keep it light" -> lower violence/gore, softer tone, more accessible pacing.
- "more intense" / "darker" -> grittier, higher tension.
- "tense but slow" -> slow-burn with high suspense (do NOT treat as a contradiction of disliking slow \
pacing — it is a narrower, tense-slow preference; update memory accordingly).

GENERAL:
- Never hallucinate movie metadata — call web_search instead of guessing.
- Always base reasoning on the LATEST living-graph state (post-update).

DISCOVERY / SURPRISE MODE (only for vague "surprise me" style asks):
- Blend TWO sources: (1) movie_hybrid_retriever candidates from the catalogue, and (2) fresh web_search \
discovery (trending / acclaimed / hidden-gem picks that fit the user's taste_profile + active likes).
- For web_search, query something like "highly rated <user's favorite genres/themes> movies to watch" or \
"trending <taste> films 2026", and pull a couple of standout titles from the answer.
- In your Final Answer, clearly SEPARATE and LABEL the two: e.g. a "🎞️ From your library" group (catalogue \
titles, each traceable to a retriever Observation) and a "🌐 Fresh picks from the web" group (web-sourced \
titles, each attributed as coming from the web, not the catalogue). Never present a web title as if it came \
from the catalogue.
- Still HONOR all active dislikes / ContentConstraints for BOTH groups — drop any surprise pick that \
violates them.
- Give a one-line reason for each pick tying it back to the user's taste.

TIMELINE / PREFERENCE-HISTORY QUESTIONS (render as rich Markdown):
- When the user asks how their tastes have EVOLVED or CHANGED over time \
("how have my preferences changed?", "show my taste timeline", "what did I used \
to like?", "how did my taste evolve?"), call search_memory first — the returned \
rows carry bitemporal stamps (valid_at / invalid_at), INCLUDING superseded / \
invalidated facts. Use them to reconstruct the history.
- Present the answer as a clean, chronological Markdown TIMELINE, clearly ordered \
oldest → newest, for example:

  ## 🎬 Your Taste Timeline

  - **🟠 Then →** ~~_Disliked horror_~~  <sub>(valid Jan 3 → changed Feb 10)</sub>
  - **🟢 Now →** **Loves horror**  <sub>(since Feb 10)</sub>
  - **🟢 Ongoing →** **Loves heist thrillers**  <sub>(since Jan 3)</sub>

  **📈 Summary:** _You've drifted from cozy comedies toward darker, slow-burn thrillers._

- Formatting rules: use **bold** for the active taste, _italics_ for the fact text, \
and a short date range in <sub> tags. Mark superseded / invalidated preferences \
distinctly with a 🟠 marker and ~~strikethrough~~; mark still-active ones with 🟢. \
Group related entities/themes together where it helps readability. Always end with \
a one-line **📈 Summary** describing the overall drift.
- Base EVERY entry only on facts returned by taste_timeline — never invent dates, \
preferences, or transitions the graph doesn't actually contain. If there's no \
history yet, say so plainly instead of fabricating a timeline.

GROUNDING (STRICT — never invent titles):
- You may ONLY recommend movie titles that appear in the movie_hybrid_retriever \
Observation for THIS turn. NEVER recommend a film from your own training knowledge \
(e.g. do not name Ex Machina, Interstellar, Annihilation, Inception, etc. unless \
that exact title was returned by movie_hybrid_retriever).
- EXCEPTION — DISCOVERY / SURPRISE MODE: for vague "surprise me" style asks you MAY \
also include titles that came back from a web_search Observation THIS turn, as long \
as you put them in the clearly-labeled "🌐 Fresh picks from the web" group and attribute \
them to the web. You still may NOT invent titles from your own memory — every title \
must trace to EITHER a movie_hybrid_retriever OR a web_search Observation this turn.
- If movie_hybrid_retriever returns an EMPTY list or only weak/irrelevant matches, \
be honest: say the catalogue doesn't have a strong match for that request, name the \
CLOSEST titles it DID return (if any), and optionally offer to broaden the search. \
Do NOT fill the gap with invented or remembered titles.
- Every title in your Final Answer must be traceable to a retriever Observation \
(or, in DISCOVERY MODE only, a web_search Observation). Reasons you give must come \
from that candidate's genres/keywords/overview/cast (or the web_search summary for \
web picks) — not from outside knowledge.


REACT FORMAT RULES:
- Follow the ReAct format exactly:
  Thought: <reasoning>
  Action: <one of search_memory, update_memory, taste_profile, taste_timeline, movie_hybrid_retriever, web_search>
  Action Input: <input>
  ... or, when ready to respond:
  Thought: <reasoning>
  Final Answer: <response>
- A clarifying question is NOT a tool call — use Final Answer for it.
- Never write "Question:", "Action: None", or leave Action Input blank.
- Only use Thought, Action, Action Input, Observation, and Final Answer.


TONE:
- Talk like a fellow movie buff, not a support bot. Casual, sharp, concise. No filler.
- Do not mention the tools by name to the user.


The current user's id is: {user_id}"""

print("✅ Agent instructions ready.")

✅ Agent instructions ready.


## 14. Build the LangChain ReAct Agent

We build a **LangChain ReAct agent** with `create_react_agent`. The five tools are bound to the current
`user_id`: `search_memory` / `update_memory` call **Graphiti** (`search` / `add_episode`), `taste_profile`
reads **Graphiti communities**, `movie_hybrid_retriever` hits the Movie KG, and `web_search` uses Tavily.


In [16]:
# ---------------------------------------------------------------------------
# LangChain ReAct agent. Tools are wrapped as single-input tools bound to the
# active user_id. search_memory / update_memory delegate to Graphiti.
# ---------------------------------------------------------------------------
from langchain_classic.agents import AgentExecutor, create_react_agent
from langchain_core.tools import Tool
from langchain_core.prompts import PromptTemplate
from langchain_core.callbacks import BaseCallbackHandler


def _build_llm():
    from langchain_openai import ChatOpenAI
    return ChatOpenAI(api_key=OPENAI_API_KEY, model=OPENAI_MODEL, temperature=0.3)


class ToolTraceHandler(BaseCallbackHandler):
    def __init__(self):
        self.trace: List[str] = []
    def on_tool_start(self, serialized, input_str, **kwargs):
        self.trace.append((serialized or {}).get("name", "tool"))


def _make_tools_for_user(user_id: str) -> List[Tool]:

    def _t_search_memory(query: str) -> str:
        # → graphiti.search()
        res = search_user_memory(user_id, query or "user preferences")
        return json.dumps(res, default=str)

    def _t_update_memory(user_message: str) -> str:
        # → graphiti.add_episode() : extraction + typing + invalidation
        res = add_user_message(user_id, user_message)
        return json.dumps(res, default=str)

    def _t_movie_hybrid_retriever(recommendation_query: str) -> str:
        cands = movie_hybrid_retriever.invoke({
            "user_id": user_id,
            "recommendation_query": recommendation_query,
            "top_k": 4,
        })
        return json.dumps(cands, default=str)

    def _t_web_search(query: str) -> str:
        title = query.split("|")[0].strip() if "|" in query else ""
        res = web_search.invoke({"query": query, "movie_title": title})
        return json.dumps(res, default=str)

    def _t_taste_profile(query: str) -> str:
        # → graphiti communities: high-level "who is this viewer" summary.
        profile = taste_profile_for_user(user_id, query or "overall movie taste")
        if not profile:
            return json.dumps({
                "taste_profile": "",
                "note": "No taste clusters yet — rely on search_memory edges "
                        "and the user's current request.",
            })
        return json.dumps({"taste_profile": profile})

    def _t_taste_timeline(query: str) -> str:
        # → Graphiti bitemporal history: active + INVALIDATED facts with dates.
        hist = search_user_memory_history(user_id, query or "")
        if not hist["rows"]:
            return json.dumps({
                "rows": [],
                "note": "No preference history yet — nothing to build a timeline from.",
            })
        return json.dumps(hist, default=str)

    return [
        Tool(name="search_memory", func=_t_search_memory,
             description="Search the user's current active movie preferences from Graphiti living "
                         "memory. Always call this FIRST before answering."),
        Tool(name="update_memory", func=_t_update_memory,
             description="Commit a user preference to Graphiti living memory by adding the message as "
                         "an episode. IMPORTANT: pass the user's ORIGINAL message text VERBATIM as "
                         "Action Input — do NOT paraphrase or rewrite it. Graphiti extracts, types, and "
                         "invalidates contradicting facts automatically."),
        Tool(name="taste_profile", func=_t_taste_profile,
             description="Get a HIGH-LEVEL summary of the user's overall taste, distilled from Graphiti "
                         "communities (Leiden clusters of their genres/themes/people). Use this for "
                         "VAGUE or open-ended requests ('surprise me', 'what should I watch tonight') "
                         "to understand the whole viewer before recommending. May be empty for new "
                         "users — then just rely on search_memory."),
        Tool(name="taste_timeline", func=_t_taste_timeline,
             description="Get the FULL bitemporal history of the user's preferences — active AND "
                         "superseded/invalidated facts with valid_at / invalid_at dates, oldest first. "
                         "Use this ONLY for timeline / preference-history questions ('how have my "
                         "tastes changed?', 'show my taste timeline', 'what did I used to like?'). "
                         "Render the result as the rich Markdown timeline described in your instructions."),
        Tool(name="movie_hybrid_retriever", func=_t_movie_hybrid_retriever,
             description="Retrieve candidate movies from the Neo4j Movie Intelligence Graph based on the "
                         "user's Graphiti preferences and the recommendation query. Use when the user "
                         "asks for a recommendation."),
        Tool(name="web_search", func=_t_web_search,
             description="Validate external/missing movie attributes (pace, violence, rating, freshness). "
                         "Input: 'movie title | question'. Use ONLY when graph metadata is incomplete."),
    ]

REACT_TEMPLATE = """
Answer the following question as best you can.

{instructions}

You have access to the following tools:
{tools}

Use this exact format:

Question: the input question you must answer
Thought: think about the next step
Action: the action to take, must be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original question

Rules:
- Output only the fields shown above.
- Do not add commentary outside this format.
- After each Thought, output either Action or Final Answer.
- Do not output both Action and Final Answer in the same step.
- If you use a tool, always provide Action Input.

Begin!

Question: {input}
{agent_scratchpad}
"""

_LLM = _build_llm()


def build_react_agent_for_user(user_id: str) -> AgentExecutor:
    tools = _make_tools_for_user(user_id)
    prompt = PromptTemplate.from_template(REACT_TEMPLATE).partial(
        instructions=AGENT_INSTRUCTIONS.format(user_id=user_id))
    react_agent = create_react_agent(_LLM, tools, prompt)
    return AgentExecutor(agent=react_agent, tools=tools, verbose=True,
                         handle_parsing_errors=True, max_iterations=8)


def _active_viewing_context(user_id: str) -> str:
    """Read the user's current WatchingFor occasion/mood from Graphiti so the
    agent can tailor tone (e.g. 'date night' -> lighter, crowd-pleasing)."""
    cypher = """
    MATCH (a:Entity)-[r:RELATES_TO {name:'WatchingFor'}]->(b:Entity)
    WHERE r.group_id = $uid
      AND r.invalid_at IS NULL AND r.expired_at IS NULL
    RETURN b.name AS context
    ORDER BY r.valid_at DESC
    LIMIT 1
    """
    try:
        with neo4j_driver.session(database=NEO4J_DATABASE) as s:
            rec = s.run(cypher, uid=user_id).single()
        return (rec["context"] if rec else "") or ""
    except Exception:
        return ""


def run_agent(user_id: str, user_message: str) -> Dict[str, Any]:
    """Run one dynamic ReAct turn. Returns {'response', 'trace'}."""
    executor = build_react_agent_for_user(user_id)
    handler = ToolTraceHandler()

    # Prepend the active viewing occasion/mood (if any) so the agent tailors tone.
    context = _active_viewing_context(user_id)
    agent_input = user_message
    if context:
        agent_input = (f"[Current viewing context: {context} — tailor tone/pick "
                       f"accordingly, do not re-ask about it]\n{user_message}")

    out = executor.invoke({"input": agent_input}, config={"callbacks": [handler]})
    return {"response": out.get("output", ""),
            "trace": " → ".join(handler.trace) or "(no tools)"}


print("✅ ReAct agent ready (Graphiti-backed memory).")

✅ ReAct agent ready (Graphiti-backed memory).


## 15. 💬 Interactive Chat UI (free-flow conversation)

Talk to the agent live. Share a preference (*"I love heist movies"*), change your mind (*"actually I'm into
slow-burn now"*), or ask for a recommendation (*"what should I watch?"*).

Each turn is handled by the ReAct agent, and the **Graphiti living memory updates in real time**. Watch the
tool trace and current preferences change as you chat, and use **Show audit trail** to see the full
bitemporal history (including invalidated facts) — all straight from Graphiti.


In [21]:
# ---------------------------------------------------------------------------
# Interactive chat UI (ipywidgets). Each turn runs the ReAct agent, then
# refreshes the live preference panel + audit trail straight from Graphiti.
# ---------------------------------------------------------------------------
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

# The active chat user (this is the group_id / user_id namespace).
CHAT_USER = "robert_williams"


def _reset_chat_memory():
    """Clear this chat user's Graphiti living memory for a fresh session.
    Removes Graphiti's episodes and its extracted entity/fact graph for this
    group_id (Graphiti-native — no manual-schema edges exist anymore)."""
    # Preferred: Graphiti's own episode removal (also unwinds derived edges).
    try:
        run_async(graphiti.remove_episodes(group_ids=[CHAT_USER]))
    except Exception as e:
        print("Reset note (remove_episodes):", e)

    # Belt-and-suspenders: drop any remaining group-scoped Graphiti nodes.
    try:
        with neo4j_driver.session(database=NEO4J_DATABASE) as s:
            s.run("""
                MATCH (n)
                WHERE n.group_id = $uid
                  AND (n:Episodic OR n:Entity OR n:Community)
                DETACH DELETE n
            """, uid=CHAT_USER)
    except Exception as e:
        print("Reset note (cleanup):", e)


# ---------------------------------------------------------------------------
# Widgets
# ---------------------------------------------------------------------------
_chat_log = widgets.Output(layout={"border": "1px solid #ddd", "height": "320px",
                                    "overflow_y": "auto", "padding": "8px"})
_prefs_out = widgets.Output()
_input = widgets.Text(placeholder="e.g. I love heist thrillers but avoid horror…",
                      layout=widgets.Layout(width="70%"))
_send_btn = widgets.Button(description="Send", button_style="primary")
_audit_btn = widgets.Button(description="Show audit trail")
_clusters_btn = widgets.Button(description="Rebuild taste clusters")
_timeline_btn = widgets.Button(description="📆 Show taste timeline", button_style="info")
_reset_btn = widgets.Button(description="Reset memory", button_style="warning")


def _refresh_prefs(show_history=False):
    with _prefs_out:
        clear_output()
        mem = search_user_memory(CHAT_USER)
        display(Markdown(f"**🧠 Current preferences:** {mem['preference_summary']}"))
        # High-level taste cluster (Graphiti communities), if any exist yet.
        profile = taste_profile_for_user(CHAT_USER)
        if profile:
            display(Markdown(f"**👥 Taste clusters:** {profile}"))
        df = inspect_user_facts(CHAT_USER, current_only=not show_history)
        if not df.empty:
            display(Markdown("**Graphiti fact edges:**"))
            display(df)


def _post(role, text):
    with _chat_log:
        who = "🧑 You" if role == "user" else "🤖 Agent"
        display(Markdown(f"**{who}:** {text}"))


def _on_send(_=None):
    msg = _input.value.strip()
    if not msg:
        return
    _input.value = ""
    _post("user", msg)
    try:
        result = run_agent(CHAT_USER, msg)
        _post("agent", result["response"])
        with _chat_log:
            display(Markdown(f"<sub>🛠️ tools: {result['trace']}</sub>"))
    except Exception as e:
        _post("agent", f"⚠️ error: {e}")
    _refresh_prefs()


def _on_audit(_=None):
    _refresh_prefs(show_history=True)


def _on_rebuild_clusters(_=None):
    with _chat_log:
        display(Markdown("*rebuilding taste clusters (Leiden + LLM summaries)…*"))
    try:
        build_taste_communities()
        with _chat_log:
            display(Markdown("*✅ taste clusters rebuilt.*"))
    except Exception as e:
        with _chat_log:
            display(Markdown(f"*⚠️ cluster rebuild error: {e}*"))
    _refresh_prefs()


def _on_reset(_=None):
    _reset_chat_memory()
    with _chat_log:
        clear_output()
        display(Markdown("*memory cleared — fresh session.*"))
    _refresh_prefs()


def _fmt_ts(val):
    """Render a Graphiti bitemporal stamp compactly (e.g. 'Feb 10')."""
    if not val:
        return ""
    try:
        return pd.to_datetime(str(val)).strftime("%b %d, %Y")
    except Exception:
        return str(val)[:10]


def _on_timeline(_=None):
    """Render the current user's FULL preference history (active + superseded)
    directly from Graphiti's bitemporal store — the same source the agent's
    taste_timeline tool reads."""
    with _chat_log:
        display(Markdown("*building taste timeline from bitemporal memory…*"))
    try:
        hist = search_user_memory_history(CHAT_USER)
    except Exception as e:
        with _chat_log:
            display(Markdown(f"*⚠️ timeline error: {e}*"))
        return

    rows = hist.get("rows", [])
    with _chat_log:
        if not rows:
            display(Markdown("*No preference history yet — nothing to show. "
                             "Chat a bit, then try again.*"))
            return

        lines = [f"## 📆 Taste Timeline for `{CHAT_USER}`", ""]
        for r in rows:
            fact = r.get("fact", "")
            valid = _fmt_ts(r.get("valid_at"))
            invalid = _fmt_ts(r.get("invalid_at") or r.get("expired_at"))
            if r.get("status") == "active":
                span = f"(since {valid})" if valid else ""
                lines.append(f"- **🟢 Now →** **_{fact}_**  <sub>{span}</sub>")
            else:
                span = (f"(valid {valid} → changed {invalid})"
                        if valid or invalid else "")
                lines.append(f"- **🟠 Then →** ~~_{fact}_~~  <sub>{span}</sub>")

        lines += ["",
                  f"**📈 Summary:** {len(hist.get('active', []))} active preference(s), "
                  f"{len(hist.get('superseded', []))} superseded."]
        display(Markdown("\n".join(lines)))


_send_btn.on_click(_on_send)
_input.on_submit(_on_send)
_audit_btn.on_click(_on_audit)
_clusters_btn.on_click(_on_rebuild_clusters)
_timeline_btn.on_click(_on_timeline)
_reset_btn.on_click(_on_reset)

display(widgets.VBox([
    _chat_log,
    widgets.HBox([_input, _send_btn]),
    widgets.HBox([_audit_btn, _clusters_btn, _timeline_btn, _reset_btn]),
    _prefs_out,
]))
_refresh_prefs()
print(f"💬 Chat ready for '{CHAT_USER}'. Type a message and hit Send.")

💬 Chat ready for 'robert_williams'. Type a message and hit Send.


> Entering new AgentExecutor chain...
Thought: The question asks about how preferences of various age groups differ, which is a general inquiry about taste trends across ages. However, I do not have direct access to aggregated data on age group preferences. My capabilities focus on the individual user's preferences and their evolution over time. I can provide insights on your own taste timeline if you want, but I cannot provide a broad demographic analysis of age group preferences.

Final Answer: I don’t have data on how preferences differ across various age groups broadly. But if you want, I can show how your own movie tastes have evolved over time or help with personalized recommendations!

> Finished chain.


> Entering new AgentExecutor chain...
Thought: To show how the user's movie taste has evolved over time, I need to retrieve their full preference history including superseded and invalidated facts from the bitem

*building taste timeline from bitemporal memory…*

## 📆 Taste Timeline for `robert_williams`

- **🟢 Now →** **_robert_williams likes classic films._**  <sub>(since Jul 23, 2026)</sub>
- **🟢 Now →** **_robert_williams likes films from the 60s._**  <sub>(since Jul 23, 2026)</sub>
- **🟢 Now →** **_robert_williams likes films from the 70s._**  <sub>(since Jul 23, 2026)</sub>
- **🟢 Now →** **_robert_williams likes films from the 80s._**  <sub>(since Jul 23, 2026)</sub>
- **🟢 Now →** **_robert_williams likes the Drama genre._**  <sub>(since Jul 23, 2026)</sub>
- **🟢 Now →** **_robert_williams likes the Action genre._**  <sub>(since Jul 23, 2026)</sub>
- **🟢 Now →** **_robert_williams likes the Thriller genre._**  <sub>(since Jul 23, 2026)</sub>
- **🟢 Now →** **_Robert Williams is especially drawn to films about India._**  <sub>(since Jul 23, 2026)</sub>
- **🟢 Now →** **_Robert Williams likes films based on true stories._**  <sub>(since Jul 23, 2026)</sub>
- **🟢 Now →** **_robert_williams is especially drawn to films about india that are based on true stories_**  <sub>(since Jul 23, 2026)</sub>
- **🟢 Now →** **_Robert Williams grew up on classic Hollywood cinema and thus has a lasting positive preference for it._**  <sub>(since Jul 23, 2026)</sub>
- **🟢 Now →** **_Robert Williams still loves timeless classics, indicating a durable positive preference for timeless classic films._**  <sub>(since Jul 23, 2026)</sub>

**📈 Summary:** 12 active preference(s), 0 superseded.

## 16. Recap — Fully Graphiti-Native

```
┌──────────────┐   search_memory   ┌─────────────────────────┐
│   👤 User     │ ────────────────▶ │ 🧠 Graphiti Living Memory │
│  message     │ ◀──────────────── │  add_episode / search    │
└──────┬───────┘   update_memory    │  (bitemporal, typed)     │
       │                            └─────────┬───────────────┘
       ▼                                       │ active preferences
┌──────────────┐  movie_hybrid_retriever ┌─────────────────────┐
│ 🤖 ReAct Agent│ ──────────────────────▶ │ 🎞️ Movie KG (GraphRAG)│
└──────┬───────┘        web_search        └─────────────────────┘
       ▼
  ✅ Personalized, audit-safe recommendation
```

| Capability | Where it lives |
|------------|----------------|
| 🧠 **Preference extraction** | `graphiti.add_episode()` LLM pipeline |
| 🏷️ **Typed entities & edges** | Custom Pydantic ontology + `edge_type_map` |
| 🔄 **Invalidation, not deletion** | Graphiti automatic bitemporal invalidation |
| 🧾 **Provenance** | Graphiti `Episodic` nodes + `MENTIONS` |
| 🔎 **Memory search** | `graphiti.search()` / `_search()` recipes |
| 👥 **Taste clusters** | `graphiti.build_communities()` |
| 🧭 **Per-user isolation** | Graphiti `group_id` namespacing |
| 🎞️ **Movie recall** | Neo4j GraphRAG `HybridRetriever` |
| 🌐 **Fresh facts on demand** | Tavily `web_search` |
| 🤖 **Orchestration** | LangChain ReAct agent |

### Try it yourself 🎬
- *"I love heist movies"* → Graphiti extracts a `Likes → heist` edge.
- *"Actually I hate horror now"* → new fact added, old one **auto-invalidated**.
- *"What should I watch?"* → grounded recommendation from the **latest** committed state.
- Click **Show audit trail** to see the full, preserved bitemporal history.


## 17. Appendix — Simulate 8 named viewers across 3 generations

To show how taste **communities differ by life-stage**, we seed a fixed roster of **8 named users** spread
across **3 generations** — **Boomers**, **Millennials**, and **Gen Z**. Each viewer has a hard-coded
`birth_year`, so their **age is derived automatically** (the user never types their age at runtime).

For every viewer we:

1. Feed a few age-appropriate, **dataset-aligned** preference messages to the agent's memory
   (`add_user_message` → Graphiti extracts + types them, isolated per `group_id`).
2. Create a dedicated **`:User` node carrying an `age` property** (plus `name`, `birth_year`,
   `generation`) and wire it into that viewer's taste graph via `HasTaste` edges — so Graphiti's Leiden
   **community detection can see the age signal** and cluster viewers by life-stage, not just genre.
3. Build taste communities per generation and print a **name + age + emergent-cluster** comparison.

> Run this at the very end — it writes 8 extra `group_id` namespaces and is independent of the live chat
> user above.


In [20]:
# ---------------------------------------------------------------------------
# Appendix: simulate 8 named viewers across 3 generations.
#
# A FIXED roster (8 users) — each with a hard-coded birth_year, so age is
# DERIVED (never typed at runtime). name / age / birth_year are SIMULATION
# CONFIG only: a real user never tells the agent those, so we do NOT feed them
# as chat messages. Instead they are written directly onto each :User node.
# For each viewer we:
#   1) seed only genuine TASTE messages (add_user_message -> Graphiti
#      extraction, isolated per group_id) — like a real chat user would,
#   2) create a :User node carrying `age` (+ name/birth_year/generation) wired
#      into the taste graph via HasTaste edges so Leiden community detection
#      sees the age signal,
#   3) build taste communities per generation and compare.
# ---------------------------------------------------------------------------
_TODAY_YEAR = utc_now().year


def _pick(profile_key: str, n: int, profile: Dict[str, List[tuple]] = None) -> List[str]:
    """Top-n best-represented catalogue values for a facet (safe if empty)."""
    profile = profile or MOVIE_GRAPH_PROFILE
    return [name for name, _ in (profile.get(profile_key) or [])[:n] if name]


# Generation metadata: era bias + generation-typical viewing flavor.
GENERATION_META = {
    "Boomers": {
        "era_hint": "classic films from the 60s, 70s and 80s",
        "flavor": [
            "I grew up on classic Hollywood cinema and I still love timeless classics.",
            "I prefer older movies over modern blockbusters.",
        ],
    },
    "Millennials": {
        "era_hint": "2000s and 2010s movies",
        "flavor": [
            "I love superhero franchises and nostalgic 2000s hits.",
            "I enjoy a good binge-worthy action adventure.",
        ],
    },
    "Gen Z": {
        "era_hint": "recent releases from the last few years",
        "flavor": [
            "I mostly watch recent releases and love animated and international films.",
            "I'm into fast-paced, visually stunning movies and anime.",
        ],
    },
}


# Fixed roster of 8 viewers across 3 generations. birth_year is hard-coded ->
# age is derived, so nothing about age is requested at runtime.
SIMULATED_USERS = [
    # user_id                name                generation      birth_year
    ("robert_williams",  "Robert Williams",  "Boomers",      1955),
    ("linda_thompson",   "Linda Thompson",   "Boomers",      1960),
    ("jessica_nguyen",   "Jessica Nguyen",   "Millennials",  1988),
    ("brandon_carter",   "Brandon Carter",   "Millennials",  1991),
    ("ashley_patel",     "Ashley Patel",     "Millennials",  1985),
    ("zoe_kim",          "Zoe Kim",          "Gen Z",        2001),
    ("ethan_garcia",     "Ethan Garcia",     "Gen Z",        2004),
    ("maya_robinson",    "Maya Robinson",    "Gen Z",        1999),
]


def _seed_messages_for_user(name: str, generation: str, birth_year: int) -> List[str]:
    """Age-appropriate, dataset-aligned PREFERENCE messages for one viewer.

    NOTE: name / age / birth_year are simulation CONFIG only — a real user never
    tells the agent those. So they are NOT sent as chat messages here; they live
    solely on the :User node (see _upsert_user_node). We only feed genuine movie
    TASTE statements, exactly like a real chat user would share."""
    meta = GENERATION_META[generation]
    genres = _pick("genres", 3)
    keywords = _pick("keywords", 2)

    messages = [
        f"I really enjoy {meta['era_hint']}.",
    ]
    if genres:
        messages.append(f"My favorite genres are {', '.join(genres)}.")
    if keywords:
        messages.append(f"I'm especially drawn to films about {', '.join(keywords)}.")
    messages.extend(meta["flavor"])
    return messages


def _upsert_user_node(user_id: str, name: str, birth_year: int, generation: str) -> None:
    """Create/refresh a :User node carrying AGE (+ name, birth_year, generation)
    and wire it into this user's taste graph. The node is also an :Entity linked
    by RELATES_TO {name:'HasTaste'} to the user's liked entities, so Graphiti's
    Leiden community detection SEES the age signal and can cluster by life-stage."""
    age = _TODAY_YEAR - birth_year
    # NOTE: this node is also labeled :Entity, so Graphiti's build_communities()
    # will load it as an EntityNode — which REQUIRES a `uuid` (string) and
    # `created_at` (datetime). Likewise the HasTaste RELATES_TO edge is loaded as
    # an EntityEdge and needs `uuid` + `created_at`. We stamp both here (only when
    # missing) so Leiden clustering doesn't fail Pydantic validation.
    cypher = """
    MERGE (u:User {group_id: $uid})
    SET u:Entity,
        u.uuid        = coalesce(u.uuid, randomUUID()),
        u.name        = $name,
        u.birth_year  = $birth_year,
        u.age         = $age,
        u.generation  = $generation,
        u.summary     = $summary,
        u.created_at  = coalesce(u.created_at, datetime()),
        u.updated_at  = datetime()
    WITH u
    MATCH (a:Entity {group_id: $uid})-[r:RELATES_TO]->(b:Entity {group_id: $uid})
    WHERE r.name IN ['Likes','Admires','PrefersEra','PrefersLanguage']
      AND r.invalid_at IS NULL AND r.expired_at IS NULL
    MERGE (u)-[hv:RELATES_TO {name: 'HasTaste', group_id: $uid}]->(b)
    SET hv.uuid = coalesce(hv.uuid, randomUUID()),
        hv.fact = $name + ' (age ' + toString($age) + ', ' + $generation + ') likes ' + b.name,
        hv.created_at = coalesce(hv.created_at, datetime())
    """
    summary = f"{name}, age {age}, {generation}. Viewer profile for taste clustering."
    with neo4j_driver.session(database=NEO4J_DATABASE) as s:
        s.run(cypher, uid=user_id, name=name, birth_year=birth_year, age=age,
              generation=generation, summary=summary)


def _build_communities_for_groups(group_ids: List[str], label: str = "") -> bool:
    """Build Graphiti communities scoped to the given group_ids. Falls back to a
    global build if this Graphiti version doesn't accept a group_ids filter.
    Never raises (a degenerate graph must not break the demo)."""
    try:
        try:
            run_async(graphiti.build_communities(group_ids=group_ids))
        except TypeError:
            run_async(graphiti.build_communities())
        print(f"   ✅ communities built for {label or group_ids}")
        return True
    except Exception as e:
        print(f"   ⚠️ community build skipped for {label or group_ids} "
              f"({type(e).__name__}: {e}).")
        return False


def simulate_generation_users(rebuild_communities: bool = True) -> Dict[str, List[str]]:
    """Seed the fixed 8-user roster into Graphiti (isolated per group_id), attach
    an age-bearing :User node to each, then build taste communities per
    generation. Returns {generation: [user_ids]}."""
    by_generation: Dict[str, List[str]] = {}

    for uid, name, generation, birth_year in SIMULATED_USERS:
        age = _TODAY_YEAR - birth_year
        print(f"\n👤 Seeding {name} [{uid}] ({generation}, born {birth_year}, age {age}):")
        for msg in _seed_messages_for_user(name, generation, birth_year):
            print("   •", msg)
            add_user_message(uid, msg)
        _upsert_user_node(uid, name=name, birth_year=birth_year, generation=generation)
        by_generation.setdefault(generation, []).append(uid)

    if rebuild_communities:
        print("\n🔨 Building taste communities per generation…")
        for generation, user_ids in by_generation.items():
            _build_communities_for_groups(user_ids, label=generation)

    return by_generation


def compare_generation_tastes(by_generation: Dict[str, List[str]]) -> pd.DataFrame:
    """Side-by-side comparison pulling each viewer's real name + age from their
    :User node and the per-user Graphiti community summary."""
    def _user_meta(uid: str) -> Dict[str, Any]:
        try:
            with neo4j_driver.session(database=NEO4J_DATABASE) as s:
                rec = s.run(
                    "MATCH (u:User {group_id:$uid}) "
                    "RETURN u.name AS name, u.age AS age", uid=uid).single()
            return dict(rec) if rec else {}
        except Exception:
            return {}

    rows = []
    for generation, user_ids in by_generation.items():
        for uid in user_ids:
            meta = _user_meta(uid)
            profile = taste_profile_for_user(uid) or "(no clusters yet)"
            signals = graphiti_signals_for_user(uid)
            rows.append({
                "generation": generation,
                "name": meta.get("name", uid),
                "age": meta.get("age"),
                "top_likes": ", ".join(signals["likes"][:5]) or "—",
                "taste_clusters": profile,
            })
    df = pd.DataFrame(rows).sort_values(["generation", "age"]).reset_index(drop=True)
    if not df.empty:
        display(Markdown("### 🧬 Taste communities across generations"))
        display(df)
    return df


# Run the 8-user simulation and compare:
_sim_by_generation = simulate_generation_users()
compare_generation_tastes(_sim_by_generation)

print("\n✅ Simulated 8 viewers across 3 generations "
      "(age-bearing :User nodes drive the community clustering).")



👤 Seeding Robert Williams [robert_williams] (Boomers, born 1955, age 71):
   • I really enjoy classic films from the 60s, 70s and 80s.
   • My favorite genres are Drama, Action, Thriller.
   • I'm especially drawn to films about india, based on true story.
   • I grew up on classic Hollywood cinema and I still love timeless classics.
   • I prefer older movies over modern blockbusters.

👤 Seeding Linda Thompson [linda_thompson] (Boomers, born 1960, age 66):
   • I really enjoy classic films from the 60s, 70s and 80s.
   • My favorite genres are Drama, Action, Thriller.
   • I'm especially drawn to films about india, based on true story.
   • I grew up on classic Hollywood cinema and I still love timeless classics.
   • I prefer older movies over modern blockbusters.

👤 Seeding Jessica Nguyen [jessica_nguyen] (Millennials, born 1988, age 38):
   • I really enjoy 2000s and 2010s movies.
   • My favorite genres are Drama, Action, Thriller.
   • I'm especially drawn to films about india, 

KeyboardInterrupt: 